# Part 0 — Setup

In [1]:
import os
import sys
from pathlib import Path
from dotenv import load_dotenv

# Ensure the project directory is in python path
project_root = Path(os.getcwd()).parent if "workspace" in os.getcwd() else Path(".").resolve()
if str(project_root) not in sys.path:
    sys.path.append(str(project_root))

# Force load environment variables from .env
env_path = project_root / ".env"
if env_path.is_file():
    load_dotenv(env_path, override=True)
    print(f"Loaded env from: {env_path}")
else:
    print("No .env file found at project root.")

# Console + file logging (writes to logs/baseline_rag.log)
from project.config import configure_logging, resolve_logs_dir

logger = configure_logging("baseline_rag")
print(f"Project root resolved to: {project_root}")
print(f"Log file: {resolve_logs_dir() / 'baseline_rag.log'}")

Loaded env from: d:\Resume-Rag\.env
Project root resolved to: d:\Resume-Rag
Log file: D:\Resume-Rag\logs\baseline_rag.log


# Part 1 — Load the 25 Resumes

In [2]:
from project.data_loader import load_documents

print("Loading documents from data/resumes...")
docs = load_documents()
print(f"Successfully loaded {len(docs)} page-level documents from the applicant pool.")

# Show unique candidates
unique_candidates = sorted(list(set(d.metadata["candidate_id"] for d in docs)))
print(f"Number of unique candidates: {len(unique_candidates)}")
print("Candidate IDs:")
for cid in unique_candidates:
    print(f" - {cid}")

d:\Resume-Rag\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
2026-09-14 13:36:26,781 - project.data_loader - INFO - Loading PDF files from D:\Resume-Rag\data\resumes
2026-09-14 13:36:26,782 - project.data_loader - INFO - Loading D:\Resume-Rag\data\resumes\adelina_project_manager.pdf


Loading documents from data/resumes...


2026-09-14 13:36:27,376 - project.data_loader - INFO - Loaded 2 page(s) from adelina_project_manager.pdf
2026-09-14 13:36:27,380 - project.data_loader - INFO - Loading D:\Resume-Rag\data\resumes\adhi_scrum_master.pdf
2026-09-14 13:36:27,790 - project.data_loader - INFO - Loaded 5 page(s) from adhi_scrum_master.pdf
2026-09-14 13:36:27,791 - project.data_loader - INFO - Loading D:\Resume-Rag\data\resumes\amar_business_systems_analyst.pdf
2026-09-14 13:36:28,806 - project.data_loader - INFO - Loaded 6 page(s) from amar_business_systems_analyst.pdf
2026-09-14 13:36:28,807 - project.data_loader - INFO - Loading D:\Resume-Rag\data\resumes\anudeep_java_developer.pdf
2026-09-14 13:36:29,192 - project.data_loader - INFO - Loaded 5 page(s) from anudeep_java_developer.pdf
2026-09-14 13:36:29,194 - project.data_loader - INFO - Loading D:\Resume-Rag\data\resumes\ashwini_j2ee_developer.pdf
2026-09-14 13:36:29,853 - project.data_loader - INFO - Loaded 8 page(s) from ashwini_j2ee_developer.pdf
2026-09

Successfully loaded 142 page-level documents from the applicant pool.
Number of unique candidates: 25
Candidate IDs:
 - adelina_project_manager
 - adhi_scrum_master
 - amar_business_systems_analyst
 - anudeep_java_developer
 - ashwini_j2ee_developer
 - avinash_project_manager
 - bapuji_hadoop_developer
 - chetan_java_developer
 - gautami_qa_mobile_testing
 - krishna_business_systems_analyst
 - mahesh_java_developer
 - mani_hadoop_developer
 - murali_pm_qa
 - navneet_business_analyst
 - nilesh_php_lead
 - pavan_java_developer
 - priyanka_business_systems_analyst
 - raju_technical_program_manager
 - ranjan_pm_scrum_master
 - ravi_pm_devops
 - shail_business_analyst
 - sharath_java_developer
 - sougandh_java_developer
 - srivatsan_project_manager
 - venkata_php_developer


# Part 2 — Chunk the Resumes

In [3]:
from project.data_loader import split_documents

print("Splitting documents into chunks...")
chunks = split_documents(docs)
print(f"Total chunks generated: {len(chunks)}")

# Inspect the first chunk and its metadata
print("\nInspecting first chunk metadata:")
import json
print(json.dumps(chunks[0].metadata, indent=2))
print("\nInspecting first chunk text preview:")
print("-" * 40)
print(chunks[0].page_content[:400] + "...")
print("-" * 40)

2026-09-14 13:36:43,023 - project.data_loader - INFO - Split 706 chunks from 142 source document(s)


Splitting documents into chunks...
Total chunks generated: 706

Inspecting first chunk metadata:
{
  "producer": "Microsoft\u00ae Word for Microsoft 365",
  "creator": "Microsoft\u00ae Word for Microsoft 365",
  "creationdate": "2024-09-19T22:43:22+03:00",
  "title": "Sample Resume for an Experienced IT Project Manager",
  "author": "Monster.com",
  "subject": "Resume Sample",
  "moddate": "2024-09-19T22:43:22+03:00",
  "source": "D:\\Resume-Rag\\data\\resumes\\adelina_project_manager.pdf",
  "total_pages": 2,
  "page": 0,
  "page_label": "1",
  "filename": "adelina_project_manager.pdf",
  "candidate_id": "adelina_project_manager",
  "type": "resume",
  "chunk_index": 0
}

Inspecting first chunk text preview:
----------------------------------------
Adelina Erimia, PMP, Six Sigma Green Belt, SMC 
[REDACTED_PHONE] ◼  [REDACTED_EMAIL] 
 
 
P R O J E C T  M A N A G E R 
 
▪ Experienced project manager offering over 10 years of successful leading diverse projects in 
the IT, Construction a

# Part 3 — Generate Embeddings

In [4]:
from project.embedding import EmbeddingManager

print("Initializing EmbeddingManager with all-MiniLM-L6-v2...")
embedding_manager = EmbeddingManager()

# Extract texts and embed them
chunk_texts = [chunk.page_content for chunk in chunks]
print(f"Generating embeddings for {len(chunk_texts)} chunks...")
embeddings = embedding_manager.generate_embeddings(chunk_texts)
print(f"Embeddings matrix shape: {embeddings.shape}")

2026-09-14 13:36:43,041 - sentence_transformers.base.model - INFO - No device provided, using cpu


Initializing EmbeddingManager with all-MiniLM-L6-v2...


2026-09-14 13:36:43,798 - httpx - INFO - HTTP Request: GET https://huggingface.co/api/agent-harnesses "HTTP/1.1 200 OK"
2026-09-14 13:36:44,031 - httpx - INFO - HTTP Request: HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2/resolve/main/modules.json "HTTP/1.1 307 Temporary Redirect"
2026-09-14 13:36:44,034 - huggingface_hub.utils._http - WARNING - Warning: You are sending unauthenticated requests to the HF Hub. Please set a HF_TOKEN to enable higher rate limits and faster downloads.
2026-09-14 13:36:44,051 - httpx - INFO - HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/sentence-transformers/all-MiniLM-L6-v2/1110a243fdf4706b3f48f1d95db1a4f5529b4d41/modules.json "HTTP/1.1 200 OK"
2026-09-14 13:36:44,280 - httpx - INFO - HTTP Request: HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2/resolve/main/config_sentence_transformers.json "HTTP/1.1 307 Temporary Redirect"
2026-09-14 13:36:44,293 - httpx - INFO - HTTP Request: HEAD https://hugg

Generating embeddings for 706 chunks...


Batches: 100%|██████████| 23/23 [00:27<00:00,  1.21s/it]
2026-09-14 13:37:17,294 - project.embedding - INFO - Embeddings generated successfully with shape (706, 384)


Embeddings matrix shape: (706, 384)


# Part 4 — Build ChromaDB

In [5]:
from project.vector_store import VectorStoreManager

print("Initializing VectorStoreManager...")
vector_store = VectorStoreManager()

print("Resetting the collection to ensure a clean baseline run...")
removed_count = vector_store.reset_collection()
print(f"Removed {removed_count} existing chunks from the collection.")

print("Adding chunks and precomputed embeddings to ChromaDB...")
added_count = vector_store.add_documents(chunks, embeddings)
print(f"Added {added_count} chunks to collection '{vector_store.collection_name}'.")
print(f"Total chunks in ChromaDB: {vector_store.get_document_count()}")

Initializing VectorStoreManager...


2026-09-14 13:37:19,370 - project.vector_store - INFO - Collection initialized: resume_baseline (706 documents)
2026-09-14 13:37:19,660 - project.vector_store - INFO - Collection initialized: resume_baseline (0 documents)
2026-09-14 13:37:19,662 - project.vector_store - INFO - Adding 706 documents to vector store
2026-09-14 13:37:20,615 - project.vector_store - INFO - Successfully added 706 documents. Total in store: 706


Resetting the collection to ensure a clean baseline run...
Removed 706 existing chunks from the collection.
Adding chunks and precomputed embeddings to ChromaDB...
Added 706 chunks to collection 'resume_baseline'.
Total chunks in ChromaDB: 706


# Part 5 — Test Baseline Retrieval

In [6]:
query = (
    "We're hiring a senior backend engineer with strong Java/J2EE experience "
    "and exposure to cloud platforms like AWS. Who would you recommend and why?"
)
print(f"Recruiter Query:\n\"{query}\"\n")

print("Generating query embedding...")
query_embedding = embedding_manager.generate_embeddings([query])

from project.config import settings

TOP_K = settings.default_top_k
print(f"Retrieving top {TOP_K} results from ChromaDB...")
retrieved_results = vector_store.similarity_search(query_embedding, top_k=TOP_K)
print(f"Retrieved {len(retrieved_results)} results.")

2026-09-14 13:37:20,636 - project.embedding - INFO - Generating embeddings for 1 texts


Recruiter Query:
"We're hiring a senior backend engineer with strong Java/J2EE experience and exposure to cloud platforms like AWS. Who would you recommend and why?"

Generating query embedding...


Batches: 100%|██████████| 1/1 [00:00<00:00, 12.03it/s]
2026-09-14 13:37:20,726 - project.embedding - INFO - Embeddings generated successfully with shape (1, 384)


Retrieving top 4 results from ChromaDB...
Retrieved 4 results.


# Part 6 — Inspect Retrieved Evidence

In [7]:
print("=== RETRIEVED EVIDENCE (MANUAL INSPECTION) ===")
for rank, result in enumerate(retrieved_results, 1):
    meta = result["metadata"]
    distance = result["distance"]
    text = result["document"]
    
    print(f"\nRank: {rank}")
    print(f"Candidate ID: {meta.get('candidate_id')}")
    print(f"Filename:     {meta.get('filename')}")
    print(f"Page:         {meta.get('page')}")
    print(f"Distance:     {distance:.4f} (Cosine)")
    print("-" * 30)
    print(text.strip())
    print("=" * 50)

=== RETRIEVED EVIDENCE (MANUAL INSPECTION) ===

Rank: 1
Candidate ID: pavan_java_developer
Filename:     pavan_java_developer.pdf
Page:         0
Distance:     0.3676 (Cosine)
------------------------------
✓ Good experience in using Amazon Web Services (AWS) and AWS SNS/SQS for delivering messages.  
✓ Expertise in using Design patterns and object oriented methodologies to design software for  Mobile 
Application.  
✓ Possess expertise in designing and developing web and mobile applications using HTML 5, CSS3, Less, Sass, 
JavaScript, JQuery, nodeJs, AngularJS . Skilled at progressing from problem statement to well documented 
designs. 
✓ Expertise in the implementation of Core Java concepts of Java, J2EE Technologies : JSP, Servlets,JSF, JSTL, EJB 
transaction implementation (CMP, BMP, Message -Driven Beans),  JMS, Struts, Spring, Swing, Hibernate, Java 
Beans, JDBC, XML,  Web Services, JNDI, Multi-Threading, Data structures etc.

Rank: 2
Candidate ID: pavan_java_developer
Filename: 

# Part 7 — Baseline LLM Generation

In [8]:
import os
from groq import Groq
from project.config import settings

# Check GROQ_API_KEY
api_key = settings.groq_api_key
if not api_key:
    print("WARNING: GROQ_API_KEY is not set. Please set it to run the LLM generation cell.")
    client = None
else:
    client = Groq(api_key=api_key)

if client:
    # Prepare prompt with retrieved evidence
    evidence_texts = []
    for rank, result in enumerate(retrieved_results, 1):
        meta = result["metadata"]
        evidence_texts.append(
            f"Evidence [{rank}] (Candidate: {meta.get('candidate_id')}, Page: {meta.get('page')}, File: {meta.get('filename')}):\n"
            f"{result['document']}"
        )
    
    context = "\n\n".join(evidence_texts)
    
    system_prompt = (
        "You are a professional recruiter assistant.\n"
        "Your job is to recommend the best candidates based ONLY on the provided retrieved resume evidence.\n"
        "CRITICAL RULES:\n"
        "1. Base your recommendation and answers ONLY on the provided evidence.\n"
        "2. If the evidence does not support a claim, do NOT make the claim.\n"
        "3. Do NOT invent, extrapolate, or assume details about the candidates. If information is missing, state that it is not present in the evidence.\n"
        "4. Be objective, precise, and professional.\n"
    )
    
    user_prompt = f"""Job Requirements:
{query}

Retrieved Evidence:
{context}

Recommendation:
Please recommend which candidate(s) match the job requirements best and explain why using the retrieved evidence. If the evidence does not provide enough information or is insufficient, explicitly state that."""

    print(f"Calling Groq LLM ({settings.groq_model_name})...")
    response = client.chat.completions.create(
        model=settings.groq_model_name,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt}
        ],
        temperature=settings.groq_temperature,
        max_tokens=settings.groq_max_tokens,
    )
    
    print("\n=== RECRUITER RECOMMENDATION FROM LLM ===")
    answer = response.choices[0].message.content
    if not (answer and answer.strip()):
        # GPT-OSS may spend tokens on reasoning; surface that if content is empty
        answer = getattr(response.choices[0].message, "reasoning", None) or "(empty response)"
    print(answer)
else:
    print("Skipping LLM call because GROQ_API_KEY is missing.")
    


Calling Groq LLM (openai/gpt-oss-20b)...


2026-09-14 13:37:22,554 - httpx - INFO - HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"



=== RECRUITER RECOMMENDATION FROM LLM ===
**Recommendation**

| Candidate | Evidence of Java/J2EE expertise | Evidence of AWS / cloud exposure | Suitability for Senior Backend Engineer |
|-----------|---------------------------------|----------------------------------|----------------------------------------|
| **pavan_java_developer** | • Core Java, J2EE (JSP, Servlets, JSF, JSTL, EJB, JMS, Struts, Spring, Hibernate, JDBC, Web Services) <br>• Design‑pattern and object‑oriented methodology experience | • “Good experience in using Amazon Web Services (AWS) and AWS SNS/SQS for delivering messages” <br>• Micro‑services with Spring Cloud, Netflix, Eureka (cloud‑native patterns) | **Highest match** – fully satisfies both Java/J2EE depth and AWS exposure. |
| **mani_hadoop_developer** | • Java, J2EE (Servlets, JSP, Java Beans, EJB, JDBC, JMS, Web Services) | • “Experience using Amazon EMR” (AWS service) | Good Java/J2EE background, but the role appears more data‑pipeline / Hadoop‑centric. A

# Part 8 — Re-ranking

### Concept

**Baseline:**
```
Query
-> embedding
-> ChromaDB
-> Top 4
```

**Problem:**
The vector similarity ranking is not always the same as task-specific relevance.

**Re-ranking:**
```
Query
-> embedding
-> ChromaDB
-> Top 15
-> Groq LLM relevance scoring
-> reorder
-> Top 4
```

### Roles in Retrieval
- **Retriever:** broad and fast candidate retrieval
- **Reranker:** second-stage relevance judgment

> **Explicit Rules:**
> - Reranking can improve ranking/precision among retrieved chunks.
> - Reranking **CANNOT** discover a chunk that was never included in the initial retrieval pool.

In [ ]:
# Query for testing Re-ranking
query = "Which candidate has hands-on experience with automated testing frameworks like Selenium?"
print(f"Recruiter Query:\n\"{query}\"\n")

# Step 1: Naive Top-4 Retrieval
query_embedding = embedding_manager.generate_embeddings([query])
naive_hits = vector_store.similarity_search(query_embedding, top_k=4)

print("========== NAIVE TOP-4 ==========")
for rank, hit in enumerate(naive_hits, 1):
    meta = hit["metadata"]
    cid = meta.get("candidate_id")
    page = meta.get("page")
    dist = hit["distance"]
    snippet = hit["document"][:200].strip()
    print(f"rank: {rank}")
    print(f"candidate: {cid}")
    print(f"page: {page}")
    print(f"vector distance: {dist:.4f}")
    print(f"snippet: {snippet}")
    print("-" * 40)

# Step 1 overfetch: Retrieve 15 candidates/chunks
overfetch_hits = vector_store.similarity_search(query_embedding, top_k=15)

# Step 2 & 3: For every retrieved chunk, ask Groq model to score from 0 to 10
print("\nScoring 15 overfetched candidate chunks using Groq LLM...")
reranked_candidates = []
for orig_rank, hit in enumerate(overfetch_hits, 1):
    meta = hit["metadata"]
    cid = meta.get("candidate_id")
    page = meta.get("page")
    dist = hit["distance"]
    doc = hit["document"]
    prompt = (
        f"How relevant is this chunk to answering the recruiter query?\n"
        f"Query: {query}\n"
        f"Chunk: {doc}\n\n"
        f"Score from 0 to 10. Output only the integer score."
    )
    try:
        if client:
            res = client.chat.completions.create(
                model=settings.groq_model_name,
                messages=[{"role": "user", "content": prompt}],
                temperature=0,
            )
            score_text = res.choices[0].message.content.strip() if res.choices else "0"
            digits = [c for c in score_text if c.isdigit()]
            score = int("".join(digits)) if digits else 0
        else:
            score = 0
    except Exception as e:
        print(f"Error scoring hit {orig_rank}: {e}")
        score = 0
    reranked_candidates.append({
        "original_rank": orig_rank,
        "candidate_id": cid,
        "page": page,
        "original_vector_distance": dist,
        "chunk_text": doc,
        "reranker_score": score
    })

# Step 4: Sort by reranker score descending
sorted_candidates = sorted(reranked_candidates, key=lambda x: (x["reranker_score"], -x["original_rank"]), reverse=True)

# Step 5: Take final top 4
top4_reranked = sorted_candidates[:4]

print("\n========== OVERFETCH + RERANK ==========")
for new_rank, item in enumerate(top4_reranked, 1):
    cid = item["candidate_id"]
    page = item["page"]
    dist = item["original_vector_distance"]
    score = item["reranker_score"]
    snippet = item["chunk_text"][:200].strip()
    print(f"new rank: {new_rank} (original rank: {item['original_rank']})")
    print(f"candidate: {cid}")
    print(f"page: {page}")
    print(f"original vector distance: {dist:.4f}")
    print(f"reranker score: {score}")
    print(f"snippet: {snippet}")
    print("-" * 40)


# Re-ranking Results

### Comparison: Naive Top-4 vs. Top-15 → Rerank → Top-4

1. **Naive Top-4:**
   - **Rank 1:** `krishna_business_systems_analyst` (page 4, vector distance 0.3144). A Business Systems Analyst resume chunk that had close cosine vector similarity.
   - **Ranks 2-4:** `gautami_qa_mobile_testing` (pages 1, 3, 2).

2. **Top-15 → Rerank → Top-4:**
   - **Promoted Relevant Evidence:** `gautami_qa_mobile_testing` (page 1) - a dedicated QA engineer detailing explicit Selenium WebDriver, TestNG, Appium, and HP ALM test automation frameworks - was promoted to **New Rank 1**.
   - **Discovered Overfetch Candidate:** `gautami_qa_mobile_testing` (page 0), originally ranked **5th** by vector distance (outside the Naive Top-4 window), received a top score of 10/10 from the Groq reranker and was promoted into the final Top 4 (New Rank 4), replacing `krishna_business_systems_analyst`.

### Conclusion
The reranker successfully promoted more relevant Selenium evidence (`gautami_qa_mobile_testing`) over keyword-adjacent vector matches (`krishna_business_systems_analyst`). Two-stage retrieval (overfetching + LLM re-ranking) improved overall ranking precision among retrieved chunks in the initial overfetch pool.

# Part 9 — Evaluation (learn by doing)

So far you built a pipeline and tried one reranking idea. **Evaluation** answers:
> *Is dense retrieval good enough? Does reranking actually help?*

We will:
1. Inspect the gold query set
2. Practice metrics on a tiny toy example
3. Score **one real query** by hand (retrieve → dedupe → metrics)
4. Run the full **dense** evaluation (no Groq needed)
5. Optionally compare **LLM rerank** on one query, then the full set

Reusable code lives in `evaluation/` and `project/reranker.py` — the notebook calls it so you can see the ideas without maintaining two copies.


## 9.0 — Concept map

```text
Gold query set (queries.json)
   query + relevant_candidates   ← human labels from real resumes
            |
            v
   Dense retrieve many CHUNKS
            |
            v
   Deduplicate by candidate_id   ← candidate-level ranking
            |
            v
   Metrics: Recall@K, Precision@K, MRR, nDCG@K
```

**Why candidate-level?** One person can appear as many chunks. Recruiters care about *people*, not duplicate pages.

**Dense vs rerank:**
- Dense: Chroma similarity only
- Rerank: overfetch chunks → Groq scores 0–10 → reorder → new top candidates


## 9.1 — Load the evaluation queries


In [ ]:
from pathlib import Path
from evaluation.evaluate import load_queries, DEFAULT_QUERIES

eval_queries = load_queries(DEFAULT_QUERIES)
print(f"Loaded {len(eval_queries)} queries from: {DEFAULT_QUERIES}")
print()
for item in eval_queries[:3]:
    print(f"{item['id']} [{item.get('difficulty')}]")
    print(f"  Q: {item['query']}")
    print(f"  Gold: {item['relevant_candidates']}")
    print()


## 9.2 — Toy metrics (no vector DB)

Imagine one query whose gold people are `A` and `B`.

Retrieved ranking (already unique candidates): `C, A, D, B`

| Metric | Intuition | Value here |
|--------|-----------|------------|
| Recall@4 | Of all gold people, how many appear in top-4? | 2/2 = 1.0 |
| Precision@4 | Of the top-4 slots, how many are gold? | 2/4 = 0.5 |
| MRR | How early is the *first* gold person? | rank 2 → 1/2 = 0.5 |
| nDCG@4 | Are gold people near the top? (order-sensitive) | between 0 and 1 |


In [ ]:
from evaluation.metrics import recall_at_k, precision_at_k, mrr, ndcg_at_k

relevant = ["A", "B"]
retrieved = ["C", "A", "D", "B"]

print("Recall@4   :", recall_at_k(relevant, retrieved, 4))
print("Precision@4:", precision_at_k(relevant, retrieved, 4))
print("MRR        :", mrr(relevant, retrieved))
print("nDCG@4     :", round(ndcg_at_k(relevant, retrieved, 4), 4))

# Try changing the order (put A first) and re-run — MRR and nDCG should improve.


## 9.3 — Prepare retriever objects

Uses the same `EmbeddingManager` / `VectorStoreManager` as Parts 3–5.
If you already ran those cells, this reuses them; otherwise it creates fresh ones.


In [ ]:
from project.embedding import EmbeddingManager
from project.vector_store import VectorStoreManager

if "embedding_manager" not in globals():
    print("Creating EmbeddingManager...")
    embedding_manager = EmbeddingManager()
else:
    print("Reusing existing embedding_manager")

if "vector_store" not in globals():
    print("Creating VectorStoreManager...")
    vector_store = VectorStoreManager()
else:
    print("Reusing existing vector_store")

print("Chunks in collection:", vector_store.get_document_count())


## 9.4 — One real query, step by step

Pick Selenium QA (`q001`). Watch:
1. raw chunk hits
2. candidate dedupe
3. metrics vs gold labels


In [ ]:
from evaluation.metrics import dedupe_candidates, recall_at_k, precision_at_k, mrr, ndcg_at_k

sample = next(q for q in eval_queries if q["id"] == "q001")
query = sample["query"]
gold = sample["relevant_candidates"]

print("Query:", query)
print("Gold candidates:", gold)
print()

# 1) Dense retrieve chunks (pool larger than top-4 so we can also talk about Recall@10)
query_embedding = embedding_manager.generate_embeddings([query])
chunk_hits = vector_store.similarity_search(query_embedding, top_k=15)

print("=== Top chunk hits (before dedupe) ===")
for i, hit in enumerate(chunk_hits[:8], 1):
    meta = hit["metadata"]
    print(f"{i:2}. {meta.get('candidate_id'):40} page={meta.get('page')} dist={hit['distance']:.4f}")

# 2) Candidate-level ranking
ranked = dedupe_candidates(chunk_hits)
print()
print("=== Unique candidates (first-seen order) ===")
for i, cid in enumerate(ranked[:10], 1):
    mark = " <-- GOLD" if cid in gold else ""
    print(f"{i:2}. {cid}{mark}")

# 3) Metrics for this single query
print()
print("Recall@1   :", recall_at_k(gold, ranked, 1))
print("Recall@4   :", recall_at_k(gold, ranked, 4))
print("Recall@10  :", recall_at_k(gold, ranked, 10))
print("Precision@4:", precision_at_k(gold, ranked, 4))
print("MRR        :", mrr(gold, ranked))
print("nDCG@4     :", round(ndcg_at_k(gold, ranked, 4), 4))


## 9.5 — Full dense evaluation (all queries, no Groq)

This loops every query in `queries.json`, retrieves, dedupes, averages metrics,
and writes `evaluation/results.json` + `evaluation/results.csv`.


In [ ]:
from evaluation.evaluate import run_evaluation

# Dense only — safe to run anytime (uses local embeddings + Chroma)
dense_summary = run_evaluation(dense_only=True)
print()
print("Dense aggregate keys:", list((dense_summary.get("dense") or {}).keys()))


## 9.6 — Optional: LLM rerank on ONE query (practice)

Same idea as Part 8, but using `project.reranker.LLMReranker`.

Requires `GROQ_API_KEY`. Skip this cell if you only want dense practice.


In [ ]:
from project.config import settings
from project.reranker import LLMReranker
from evaluation.metrics import dedupe_candidates, precision_at_k, mrr, ndcg_at_k

if not settings.groq_api_key:
    print("GROQ_API_KEY missing — skip this cell or set it in .env")
else:
    sample = next(q for q in eval_queries if q["id"] == "q001")
    query = sample["query"]
    gold = sample["relevant_candidates"]

    query_embedding = embedding_manager.generate_embeddings([query])
    pool = vector_store.similarity_search(query_embedding, top_k=15)
    dense_ranked = dedupe_candidates(pool)

    reranker = LLMReranker()
    # Score the whole pool, then take candidate-level top-4 after dedupe
    reranked_hits = reranker.rerank(query, pool, top_k=len(pool))
    rerank_ranked = dedupe_candidates(reranked_hits)

    print("Dense top-4   :", dense_ranked[:4])
    print("Rerank top-4  :", rerank_ranked[:4])
    print("Gold          :", gold)
    print()
    print("Dense  P@4 / MRR / nDCG@4:",
          round(precision_at_k(gold, dense_ranked, 4), 3),
          round(mrr(gold, dense_ranked), 3),
          round(ndcg_at_k(gold, dense_ranked, 4), 3))
    print("Rerank P@4 / MRR / nDCG@4:",
          round(precision_at_k(gold, rerank_ranked, 4), 3),
          round(mrr(gold, rerank_ranked), 3),
          round(ndcg_at_k(gold, rerank_ranked, 4), 3))


## 9.7 — Optional: full dense + rerank comparison

Runs all queries with LLM scoring (~15 Groq calls per query). Slow and rate-limited.
Prefer 9.5 for everyday practice; use this when you want a full compare.


In [ ]:
from project.config import settings
from evaluation.evaluate import run_evaluation

if not settings.groq_api_key:
    print("GROQ_API_KEY missing — run Part 9.5 dense-only instead.")
else:
    # WARNING: many Groq calls. Uncomment to run:
    # full_summary = run_evaluation(dense_only=False)
    print("Left commented on purpose. Uncomment the line above when you want a full rerank eval.")


## 9.8 — Read saved results

After 9.5 (or 9.7), inspect the summary file the evaluator wrote.


In [ ]:
import json
from pathlib import Path

results_path = Path("..") / "evaluation" / "results.json"
if not results_path.is_file():
    results_path = Path("evaluation") / "results.json"

data = json.loads(results_path.read_text(encoding="utf-8"))
print("File:", results_path.resolve())
print(json.dumps(data["summary"], indent=2))

print()
print("Example per-query row (first query):")
row = data["queries"][0]
print("id:", row["id"])
print("query:", row["query"])
print("gold:", row.get("relevant_candidates"))
print("dense top candidates:", row.get("dense_retrieved_candidates"))
print("dense metrics:", row.get("dense_metrics"))


## Takeaways

1. **Gold labels** turn “looks good” into numbers you can compare.
2. **Dedupe by `candidate_id`** before metrics — otherwise one person floods the top-K.
3. Dense is strong at **finding** people (high Recall@10); weaker at **ranking** the best few (Precision@4 / nDCG).
4. Reranking only reorders the retrieved pool — it cannot invent a missed resume.
5. Day-to-day: practice in this notebook; for scripts/CI use `python -m evaluation.evaluate`.
